# E1 — Longitudinal calibration reuse

A three-class classifier (air, acetone, pentane) is calibrated on an early period of campaign 130 and evaluated without updates on later measurements from the same device. Concentration-matched conditions and a later recalibration provide controls. Concentrations are used for cohort matching, not regression.

## 1. Dataset and fixed settings

Configure `ESMELL_DATA_DIR`. Training and validation occur in September; the following dates are held out.

In [ ]:
from pathlib import Path
import os
import copy
import random
import re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from esdata import EsmellDataset, Chain
from esdata.transforms import (
    DefaultCSVPreprocessing, SimplifyLabels, SelectRChannels,
    UserLabelCycleSegments, CoolingHeatingSegments,
    CanonicalTempAlign, CanonicalSubtractionFeatures,
)

REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "src" / "esdata").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Benchmark repository root not found")
DATA_DIR = Path(os.environ.get("ESMELL_DATA_DIR", str(REPO_ROOT / "data" / "raw")))
OUT_DIR = REPO_ROOT / "outputs" / "E1_unified"
CARDS = (130, 146, 166, 169)
CLASSES = ("air", "acetone", "pentane")
LEVELS = {
    "air": (0,),
    "acetone": (328, 655, 1310, 2620, 5240),
    "pentane": (438, 875, 1750, 3500),
}
# E1: fit on campaign 130; later campaigns are test-only.
TRAIN_START = pd.Timestamp("2026-09-18")
VAL_START = pd.Timestamp("2026-09-26")
TEST_START = pd.Timestamp("2026-09-27")
END_130 = pd.Timestamp("2026-09-29")
# Later controls: early versus late periods within each campaign.
CUTOFF = {146: pd.Timestamp("2026-10-07"),
          166: pd.Timestamp("2026-10-16"),
          169: pd.Timestamp("2026-10-19")}
START_169, END_169 = pd.Timestamp("2026-10-18"), pd.Timestamp("2026-10-20")
SEED, RANDOM_REPEATS = 42, 3
EPOCHS, PATIENCE, BATCH_SIZE = 60, 12, 32
MIN_CONDITION_N = 3
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

## 2. Sensor features

Apply the original `esdata` processing to six physical channels, yielding 12 temperature-aligned heating/hysteresis curves per recording.

In [ ]:
HEAT, COOL = (1, -3, 5), (0, -2, 4)
CHANNELS = ("R1", "R2", "R11", "R12", "R16", "R17")
EXCLUDE = ("R3", "R4", "R5", "R6", "R7", "R8", "R9", "R10", "R13", "R14", "R15")
LABEL_RE = re.compile(r"([a-z]+)\[([\d.]+)(ppb|ppm)\]", re.I)
CLASS_ID = {name: i for i, name in enumerate(CLASSES)}
EXPECTED = {(gas, ppb) for gas, levels in LEVELS.items() for ppb in levels}


def condition(label):
    if label in ("air", "none"):
        return "air", 0
    match = LABEL_RE.fullmatch(label)
    if match is None:
        return None  # mixtures or gases outside the task vocabulary
    gas, amount, unit = match.groups()
    ppb = round(float(amount) * (1000 if unit.lower() == "ppm" else 1))
    return (gas, ppb) if (gas, ppb) in EXPECTED else None


def role_of(card, when):
    if card == 130:
        if TRAIN_START <= when < VAL_START:
            return "train"
        if VAL_START <= when < TEST_START:
            return "validation"
        if TEST_START <= when < END_130:
            return "test"
        return None
    if card == 169 and not (START_169 <= when < END_169):
        return None
    return "test"


rows, features, errors, protocols = [], [], [], []
for card in CARDS:
    source = EsmellDataset(
        root_path=str(DATA_DIR), subfolders=f"curiosity-2v15.{card}.0.tar.gz",
        csv_repair="strict", archive_cache="disk", preload=True,
    )
    pipeline = source >> Chain(
        DefaultCSVPreprocessing(), SimplifyLabels(),
        SelectRChannels(exclude=EXCLUDE),
        UserLabelCycleSegments(),
        CoolingHeatingSegments(temp_col="T_mean", smooth_window=3, eps=0.2),
        CanonicalTempAlign(n_grid=200, temp_range=(175., 390.),
                           normalize=False, conductance=True),
        CanonicalSubtractionFeatures(features=(HEAT, (HEAT, COOL))),
    )
    assert tuple(pipeline.resistance_channels) == CHANNELS
    info = next(iter(source.experiment_cards.values()))[card]
    protocols.append({"card": card, "chip": info.get("chip_id"),
                      "stand": info.get("stand_id"), "profile": info.get("thermo_profile"),
                      "flow_ml_min": info.get("flow", {}).get("value"),
                      "baseline_channel": info.get("baseline_modes", {}).get("target_channel"),
                      "stabilization_points": info.get("baseline_modes", {}).get("stabilization_points")})
    for sid in tqdm(source.ids, desc=f"Card {card}"):
        m = source.metadata(sid)
        when = pd.Timestamp(m["datetime"])
        parsed, role = condition(m["classname"]), role_of(card, when)
        if parsed is None or role is None:
            continue
        sample_id = f"{m['domain']}::{m['filename']}"
        try:
            x = np.asarray(pipeline.features(sid), dtype=np.float32)
            if x.shape != (12, 200) or not np.isfinite(x).all() or not x.any():
                raise ValueError(f"Invalid features: {x.shape}")
        except Exception as exc:
            errors.append({"card": card, "sample_id": sample_id, "reason": str(exc)[:200]})
            continue
        gas, ppb = parsed
        rows.append({"card": card, "sample_id": sample_id, "datetime": when,
                     "gas": gas, "ppb": ppb, "role": role})
        features.append(x)
    print(f"{card}: {sum(r['card'] == card for r in rows)} included; "
          f"{sum(e['card'] == card for e in errors)} preprocessing errors")

meta = pd.DataFrame(rows)
if meta.empty:
    raise RuntimeError("No samples: check DATA_DIR and metadata")
X = np.stack(features)
y = meta.gas.map(CLASS_ID).to_numpy(dtype=np.int64)
assert meta.sample_id.is_unique and X.shape == (len(meta), 12, 200)
protocols = pd.DataFrame(protocols)
meta.to_csv(OUT_DIR / "manifest.csv", index=False)
protocols.to_csv(OUT_DIR / "protocols.csv", index=False)
pd.DataFrame(errors, columns=["card", "sample_id", "reason"]).to_csv(OUT_DIR / "qc_errors.csv", index=False)

## 3. Cohort checks

Restrict the task to ten shared air/gas–concentration conditions. Verify that each evaluation cohort contains the required conditions.

In [ ]:
assert protocols.chip.nunique() == protocols.stand.nunique() == protocols.profile.nunique() == 1
late_settings = protocols.set_index("card").loc[[166, 169]]
assert late_settings.stabilization_points.tolist() == [5, 5]
assert late_settings.flow_ml_min.nunique() == late_settings.baseline_channel.nunique() == 1
print("Protocols:\n", protocols.to_string(index=False))

counts = (meta.assign(day=meta.datetime.dt.date)
          .groupby(["card", "day", "gas", "ppb"]).size().rename("n").reset_index())
counts.to_csv(OUT_DIR / "condition_counts.csv", index=False)
print("\nSamples:\n", meta.groupby(["card", "role", "gas"]).size().unstack(fill_value=0).to_string())


def check_conditions(indices, name, min_n=1):
    observed = meta.iloc[indices].groupby(["gas", "ppb"]).size()
    if set(observed.index) != EXPECTED or observed.min() < min_n:
        raise ValueError(f"{name}: incomplete gas/concentration coverage")


train = np.flatnonzero(meta.role.eq("train"))
val = np.flatnonzero(meta.role.eq("validation"))
test = np.flatnonzero(meta.role.eq("test"))
for name, ids in (("130 train", train), ("130 val", val)):
    check_conditions(ids, name)
assert len(train) and len(val) and len(test)
assert meta.datetime.iloc[train].max() < meta.datetime.iloc[val].min()
assert meta.datetime.iloc[val].max() < meta.datetime.iloc[test].min()

# (experiment, protocol, train_card, test_card, repeat, train_idx, test_idx)
controls = []
for card in (146, 166, 169):
    ids = np.flatnonzero(meta.card.eq(card))
    early = ids[(meta.datetime.iloc[ids] < CUTOFF[card]).to_numpy()]
    late = ids[(meta.datetime.iloc[ids] >= CUTOFF[card]).to_numpy()]
    for role, ix in (("train", early), ("test", late)):
        check_conditions(ix, f"{card} future {role}", min_n=3)
    controls.append((f"{card}_future", "within_future", card, card, 0, early, late))

    strata = (meta.gas.iloc[ids] + ":" + meta.ppb.iloc[ids].astype(str)).to_numpy()
    splitter = StratifiedShuffleSplit(n_splits=RANDOM_REPEATS,
                                      test_size=len(late), random_state=SEED)
    for rep, (tr, te) in enumerate(splitter.split(ids, strata), 1):
        for role, ix in (("train", ids[tr]), ("test", ids[te])):
            check_conditions(ix, f"{card} random {rep} {role}", min_n=3)
        controls.append((f"{card}_random_{rep}", "within_random", card, card,
                         rep, ids[tr], ids[te]))

ix166 = np.flatnonzero(meta.card.eq(166))
ix169 = np.flatnonzero(meta.card.eq(169))
assert meta.datetime.iloc[ix166].max() < meta.datetime.iloc[ix169].min()
check_conditions(ix166, "166 train", min_n=3)
check_conditions(ix169, "169 test", min_n=3)
controls.append(("166_to_169", "cross_session", 166, 169, 0, ix166, ix169))
print(f"\nInitial: train={len(train)}, val={len(val)}, future test={len(test)}")
print("Controls:", len(controls), "(3 future + 9 random + 1 cross-session)")

## 4. Models and evaluation

A 1D CNN and ExtraTrees form the frozen calibration. Later controls use ExtraTrees. Report balanced accuracy, macro-F1, component recall, and condition-balanced accuracy.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class ConvBlock1D(nn.Module):
    def __init__(self, input_channels, output_channels, kernel, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_channels, output_channels, kernel, padding=kernel // 2),
            nn.BatchNorm1d(output_channels), nn.ReLU(inplace=True),
            nn.Conv1d(output_channels, output_channels, kernel, padding=kernel // 2),
            nn.BatchNorm1d(output_channels), nn.ReLU(inplace=True),
            nn.MaxPool1d(2), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class RobustEsmellNet(nn.Module):
    def __init__(self, n_channels):
        super().__init__()
        self.backbone = nn.Sequential(
            ConvBlock1D(n_channels, 64, 7, .10),
            ConvBlock1D(64, 128, 5, .15),
            ConvBlock1D(128, 256, 3, .20),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(.30), nn.Linear(256, 3),
        )

    def forward(self, x):
        h = self.backbone(x)
        return self.classifier(torch.cat((h.mean(-1), h.amax(-1)), dim=1))


def statistics(x):
    blocks = [x.mean(2), x.std(2), x.min(2), x.max(2),
              np.median(x, axis=2), np.quantile(np.abs(x), .9, axis=2)]
    blocks.extend(part.mean(2) for part in np.array_split(x, 10, axis=2))
    return np.concatenate(blocks, axis=1)


def score(frame):
    correct = frame.true.eq(frame.pred)
    counts = frame.groupby(["gas", "ppb"]).size()
    full = set(counts.index) == EXPECTED and counts.min() >= MIN_CONDITION_N
    by_condition = frame.assign(correct=correct).groupby(["gas", "ppb"]).correct.mean()
    all_classes = set(frame.true) == set(range(3))
    return {
        "n_test": len(frame), "n_conditions": len(counts),
        "bacc": balanced_accuracy_score(frame.true, frame.pred) if all_classes else np.nan,
        "condition_bacc": by_condition.groupby(level=0).mean().mean() if full else np.nan,
        "macro_f1": f1_score(frame.true, frame.pred, labels=range(3),
                              average="macro", zero_division=0) if all_classes else np.nan,
        "pentane_recall": correct[frame.gas.eq("pentane")].mean(),
        "min_condition_n": int(counts.min()),
        "pred_classes": frame.pred.nunique(),
    }


def make_predictions(experiment, protocol, model, train_card, test_idx, proba, repeat=0):
    frame = meta.iloc[test_idx].copy().reset_index(drop=True)
    frame["true"] = y[test_idx]
    frame["pred"] = proba.argmax(1)
    frame["pred_gas"] = [CLASSES[i] for i in frame.pred]
    frame["confidence"] = proba.max(1)
    frame["experiment"] = experiment
    frame["protocol"] = protocol
    frame["model"] = model
    frame["train_card"] = train_card
    frame["test_card"] = frame.card
    frame["repeat"] = repeat
    return frame


Z = statistics(X)  # compute per-record statistics independently
prediction_frames, split_rows = [], []


def save_split(experiment, role, indices):
    split_rows.extend({"experiment": experiment, "role": role,
                       "sample_id": meta.sample_id.iloc[i]} for i in indices)

## 5. Frozen calibration

Select the CNN checkpoint on the original validation period and evaluate without adaptation on the later campaigns.

In [ ]:
set_seed(SEED)
mean = X[train].mean((0, 2), keepdims=True)
std = X[train].std((0, 2), keepdims=True)
std = np.where(std < 1e-6, 1., std)


def loader(indices, shuffle=False):
    xb = torch.tensor((X[indices] - mean) / std, dtype=torch.float32)
    yb = torch.tensor(y[indices], dtype=torch.long)
    return DataLoader(TensorDataset(xb, yb), batch_size=BATCH_SIZE, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(SEED))


def cnn_probabilities(model, indices):
    model.eval()
    batches = []
    with torch.no_grad():
        for xb, _ in loader(indices):
            batches.append(torch.softmax(model(xb.to(DEVICE)), dim=1).cpu().numpy())
    return np.concatenate(batches)


cnn = RobustEsmellNet(X.shape[1]).to(DEVICE)
optimizer = torch.optim.AdamW(cnn.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=.5, patience=3)
train_loader = loader(train, shuffle=True)
history, best, best_epoch, state, stale = [], -1., 0, None, 0
for epoch in range(1, EPOCHS + 1):
    cnn.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        nn.functional.cross_entropy(cnn(xb), yb).backward()
        optimizer.step()
    pred_val = cnn_probabilities(cnn, val).argmax(1)
    val_bacc = balanced_accuracy_score(y[val], pred_val)
    history.append({"epoch": epoch, "val_bacc": val_bacc})
    scheduler.step(val_bacc)
    if val_bacc > best:
        best, best_epoch, state, stale = val_bacc, epoch, copy.deepcopy(cnn.state_dict()), 0
    else:
        stale += 1
        if stale >= PATIENCE:
            break
cnn.load_state_dict(state)
pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)
torch.save({"state_dict": cnn.cpu().state_dict(), "mean": mean, "std": std,
            "classes": CLASSES, "best_epoch": best_epoch}, OUT_DIR / "cnn.pt")
cnn.to(DEVICE)

forest = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2,
                              class_weight="balanced", n_jobs=-1, random_state=SEED)
forest.fit(Z[np.r_[train, val]], y[np.r_[train, val]])
assert forest.classes_.tolist() == list(range(3))
joblib.dump(forest, OUT_DIR / "extra_trees.joblib")

save_split("frozen_130", "train", train)
save_split("frozen_130", "validation", val)
save_split("frozen_130", "test", test)
prediction_frames += [
    make_predictions("frozen_130", "frozen", "CNN", 130, test,
                     cnn_probabilities(cnn, test)),
    make_predictions("frozen_130", "frozen", "ExtraTrees", 130, test,
                     forest.predict_proba(Z[test])),
]
print(f"Frozen CNN: epoch {best_epoch}, validation BAcc={best:.3f}; "
      f"ExtraTrees trained on {len(train) + len(val)} samples")

## 6. Recalibration controls

Assess within-campaign random and forward-time splits and a later-to-later campaign transfer. These are distinct diagnostics rather than an additional model search.

In [ ]:
for experiment, protocol, source_card, test_card, repeat, tr, te in tqdm(controls):
    assert len(tr) and len(te) and not np.intersect1d(tr, te).size
    model = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2,
                                 class_weight="balanced", n_jobs=-1, random_state=SEED)
    model.fit(Z[tr], y[tr])
    assert model.classes_.tolist() == list(range(3))
    prediction_frames.append(make_predictions(experiment, protocol, "ExtraTrees",
                                              source_card, te, model.predict_proba(Z[te]), repeat))
    save_split(experiment, "train", tr)
    save_split(experiment, "test", te)

predictions = pd.concat(prediction_frames, ignore_index=True)
predictions.to_csv(OUT_DIR / "predictions.csv", index=False)
pd.DataFrame(split_rows).to_csv(OUT_DIR / "splits.csv", index=False)

result_rows, recall_rows, daily_rows = [], [], []
for (exp, protocol, model, test_card), group in predictions.groupby(
        ["experiment", "protocol", "model", "test_card"], sort=False):
    result_rows.append({"experiment": exp, "protocol": protocol, "model": model,
                        "test_card": test_card, **score(group)})
    for (gas, ppb), part in group.groupby(["gas", "ppb"]):
        recall_rows.append({"experiment": exp, "model": model, "test_card": test_card,
                            "gas": gas, "ppb": ppb, "n": len(part),
                            "recall": part.true.eq(part.pred).mean()})
    if exp == "frozen_130":
        for day, part in group.groupby(group.datetime.dt.date):
            daily_rows.append({"day": day, "card": test_card, "model": model, **score(part)})

summary = pd.DataFrame(result_rows)
fit_sizes = {("frozen_130", "CNN"): len(train),
             ("frozen_130", "ExtraTrees"): len(train) + len(val)}
fit_sizes.update({(exp, "ExtraTrees"): len(tr) for exp, _, _, _, _, tr, _ in controls})
summary["n_train_fit"] = [fit_sizes[(e, m)] for e, m in zip(summary.experiment, summary.model)]
summary.to_csv(OUT_DIR / "summary.csv", index=False)
pd.DataFrame(recall_rows).to_csv(OUT_DIR / "condition_recalls.csv", index=False)
class_recalls = (predictions.assign(correct=predictions.true.eq(predictions.pred))
                 .groupby(["experiment", "protocol", "model", "test_card", "gas"], as_index=False)
                 .agg(n=("correct", "size"), recall=("correct", "mean")))
class_recalls.to_csv(OUT_DIR / "class_recalls.csv", index=False)
random_summary = (summary[summary.protocol.eq("within_random")]
                  .groupby(["model", "test_card"], as_index=False)
                  .agg(runs=("bacc", "size"), bacc_mean=("bacc", "mean"),
                       bacc_sd=("bacc", "std"),
                       pentane_recall_mean=("pentane_recall", "mean"),
                       pentane_recall_sd=("pentane_recall", "std")))
random_summary.to_csv(OUT_DIR / "random_summary.csv", index=False)
pd.DataFrame(daily_rows).to_csv(OUT_DIR / "daily_scores.csv", index=False)
print("\nFrozen calibration:\n", summary[summary.experiment.eq("frozen_130")][
    ["model", "test_card", "n_test", "bacc", "condition_bacc", "pentane_recall"]
].round(3).to_string(index=False))
print("\nLate controls:\n", summary[summary.protocol.ne("frozen")][
    ["experiment", "n_test", "bacc", "pentane_recall"]
].round(3).to_string(index=False))

## 7. Matched evaluation on the final campaign

Compare the original and recently trained classifiers on precisely the same 275 measurements from campaign 169.

In [ ]:
late = predictions[predictions.test_card.eq(169)]
comparisons = [
    ("130 CNN", "frozen_130", "CNN"),
    ("130 ExtraTrees", "frozen_130", "ExtraTrees"),
    ("166 ExtraTrees", "166_to_169", "ExtraTrees"),
]
matched = None
for label, experiment, model in comparisons:
    part = late[late.experiment.eq(experiment) & late.model.eq(model)]
    cols = part[["sample_id", "gas", "ppb", "true", "pred"]].copy()
    cols = cols.rename(columns={"pred": label})
    if matched is None:
        matched = cols
    else:
        matched = matched.merge(cols[["sample_id", label]], on="sample_id",
                                how="inner", validate="one_to_one")

expected_ids = set(late[late.experiment.eq("166_to_169")].sample_id)
assert len(matched) == len(expected_ids) and set(matched.sample_id) == expected_ids
assert len(matched) == len(meta[meta.card.eq(169)])
matched.to_csv(OUT_DIR / "matched_169.csv", index=False)

matched_scores = []
for label, _, _ in comparisons:
    correct = matched.true.eq(matched[label])
    matched_scores.append({"model": label, "n_test": len(matched),
                           "bacc": balanced_accuracy_score(matched.true, matched[label]),
                           "pentane_recall": correct[matched.gas.eq("pentane")].mean(),
                           "pentane_correct": int(correct[matched.gas.eq("pentane")].sum()),
                           "pentane_total": int(matched.gas.eq("pentane").sum())})
matched_scores = pd.DataFrame(matched_scores)
matched_scores.to_csv(OUT_DIR / "matched_169_scores.csv", index=False)
print("Matched test on 169:\n", matched_scores.round(3).to_string(index=False))

# Figure 1: frozen accuracy over sessions + pentane recovery on identical 169 samples.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.7), constrained_layout=True)
short = summary[summary.experiment.eq("frozen_130")]
for model, part in short.groupby("model"):
    part = part.set_index("test_card").reindex(CARDS)
    ax1.plot(range(len(CARDS)), part.condition_bacc, "o--", label=model)
ax1.axhline(1 / 3, ls=":", color="gray", label="Chance (1/3)")
ax1.set(xticks=range(len(CARDS)),
        xticklabels=["130\ns=100", "146\ns=10", "166\ns=5", "169\ns=5"],
        ylim=(0, 1.03), ylabel="Condition-balanced BAcc",
        title="Frozen calibration trained on 130")
ax1.legend(fontsize=8)
ax1.grid(alpha=.2)
ax2.bar(matched_scores.model, matched_scores.pentane_recall)
ax2.set(ylim=(0, 1.03), ylabel="Pentane recall",
        title=f"Same {len(matched)} samples of card 169")
ax2.tick_params(axis="x", labelrotation=20)
ax2.grid(axis="y", alpha=.2)
fig.savefig(OUT_DIR / "fig_E1_main.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "fig_E1_main.pdf", bbox_inches="tight")
plt.show()

## 8. Signal diagnostics

Plot representative temperature-response curves to describe feature shift; these do not establish a physical cause of aging.

In [ ]:
strengths, shifts = [], []
train_meta = meta.iloc[train]
for card in CARDS:
    ids = np.flatnonzero(meta.card.eq(card) & (meta.role.eq("test") if card == 130 else True))
    air_ids = ids[(meta.gas.iloc[ids] == "air").to_numpy()]
    if not len(air_ids):
        continue
    air = np.median(X[air_ids], axis=0)
    for gas, ppb in sorted(EXPECTED):
        sample_ids = ids[(meta.gas.iloc[ids].eq(gas) & meta.ppb.iloc[ids].eq(ppb)).to_numpy()]
        if not len(sample_ids):
            continue
        median = np.median(X[sample_ids], axis=0)
        strengths.append({"card": card, "gas": gas, "ppb": ppb, "n": len(sample_ids),
                          "rms_vs_air": float(np.sqrt(np.mean((median - air) ** 2)))})
        ref_ids = train[(train_meta.gas.eq(gas) & train_meta.ppb.eq(ppb)).to_numpy()]
        ref = np.median(X[ref_ids], axis=0)
        shifts.append({"card": card, "gas": gas, "ppb": ppb,
                       "rms_vs_initial": float(np.sqrt(np.mean((median - ref) ** 2)))})

pd.DataFrame(strengths).to_csv(OUT_DIR / "signal_strength.csv", index=False)
pd.DataFrame(shifts).to_csv(OUT_DIR / "feature_shift.csv", index=False)

# Figure 2: representative temperature curves, identical gases and concentrations.
T = np.linspace(175, 390, 200)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
for ax, (gas, ppb) in zip(axes, (("acetone", 1310), ("pentane", 1750))):
    ref_ids = train[(train_meta.gas.eq(gas) & train_meta.ppb.eq(ppb)).to_numpy()]
    ax.plot(T, np.median(X[ref_ids, 0, :], axis=0), lw=2, label="130 · train")
    for card in CARDS:
        ids = np.flatnonzero(meta.card.eq(card) & meta.role.eq("test") &
                             meta.gas.eq(gas) & meta.ppb.eq(ppb))
        if len(ids) >= 3:
            ax.plot(T, np.median(X[ids, 0, :], axis=0), label=f"{card} · test")
    ax.set(xlabel="Temperature (°C)", title=f"{gas} · {ppb} ppb")
    ax.grid(alpha=.2)
axes[0].set_ylabel("R1 · baseline-referenced log conductance")
axes[1].legend(fontsize=8)
fig.savefig(OUT_DIR / "fig_E1_signals.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "fig_E1_signals.pdf", bbox_inches="tight")
plt.show()

# Diagnostic confusion matrix: recalibrated model 166 -> 169.
part = predictions[predictions.experiment.eq("166_to_169")]
cm = confusion_matrix(part.true, part.pred, labels=range(3), normalize="true")
fig, ax = plt.subplots(figsize=(4.6, 3.8), constrained_layout=True)
img = ax.imshow(cm, vmin=0, vmax=1)
ax.set(xticks=range(3), yticks=range(3), xticklabels=CLASSES,
       yticklabels=CLASSES, xlabel="Predicted", ylabel="True", title="166 → 169")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center")
fig.colorbar(img, ax=ax, fraction=.046)
fig.savefig(OUT_DIR / "fig_E1_confusion.png", dpi=200, bbox_inches="tight")
plt.show()

## Outputs

The exported scores, per-condition recalls, predictions, protocol manifest, and PDF figures support the E1 results. The principal comparisons are the frozen calibration across campaigns and the same-record recalibration control.